# Deploy EmbeddingGemma on SageMaker AI with the vLLM container

This notebook deploys [`google/embeddinggemma-300m`](https://huggingface.co/google/embeddinggemma-300m) — a 300M parameter multilingual text embedding model from Google — to a **SageMaker AI real-time endpoint** using the **AWS Deep Learning Container for vLLM**.

## What's covered
- Building the model + endpoint using the vLLM SageMaker container (`server-sagemaker-cuda` variant)
- Passing `SM_VLLM_*` environment variables to configure the vLLM engine as a pooling (embedding) model
- Calling the OpenAI-compatible `/v1/embeddings` schema through `invoke_endpoint`
- Using EmbeddingGemma's task-specific prompt prefixes (query vs. document)
- Truncating embeddings to lower dimensions via Matryoshka Representation Learning (MRL)

## References
- [EmbeddingGemma model card](https://huggingface.co/google/embeddinggemma-300m)
- [vLLM DLC — SageMaker AI deployment](https://aws.github.io/deep-learning-containers/vllm/deployment/sagemaker/)
- [vLLM DLC — configuration reference](https://aws.github.io/deep-learning-containers/vllm/configuration/)
- [Available Deep Learning Container images](https://aws.github.io/deep-learning-containers/reference/available_images/#vllm-ubuntu)

## Before you start
`google/embeddinggemma-300m` is a **gated model**. You must:
1. Log in to Hugging Face and accept Google's Gemma usage license on the [model page](https://huggingface.co/google/embeddinggemma-300m).
2. Create a Hugging Face access token (read scope is enough) at https://huggingface.co/settings/tokens.

You'll be prompted to paste that token securely below (it is not hardcoded or written to disk).

In [ ]:
import json
import time
import re
import getpass

import boto3

region = boto3.Session().region_name or "us-east-1"

sm = boto3.client("sagemaker", region_name=region)
sm_runtime = boto3.client("sagemaker-runtime", region_name=region)

print(f"Region: {region}")

In [ ]:
from IPython.display import clear_output
import boto3


def get_sagemaker_role():
    iam = boto3.client("iam")
    sts = boto3.client("sts")
    assumed_role_arn = sts.get_caller_identity()["Arn"]
    # Extract the role name from the assumed-role ARN (strips session suffix)
    # assumed-role ARN format: arn:aws:sts::<account>:assumed-role/<role-name>/<session>
    role_name = assumed_role_arn.split(":assumed-role/")[1].split("/")[0]
    # Look up the actual role ARN via IAM (preserves service-role/ path prefix)
    role = iam.get_role(RoleName=role_name)["Role"]
    return role["Arn"]


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 30):
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    while status in ("Creating", "Updating"):
        time.sleep(sleep_time)
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        clear_output(wait=True)
        progress += "."
        print(progress)
    print(f"Endpoint '{endpoint_name}' status: '{status}'")
    if status != "InService":
        info = sm.describe_endpoint(EndpointName=endpoint_name)
        print(info.get("FailureReason", "No failure reason returned."))
    return status


# Overwrite with your role ARN if running outside of SageMaker Studio / a notebook instance
role = None
if role is None:
    role = get_sagemaker_role()
print(role)

In [ ]:
# Paste your Hugging Face access token (read scope). Input is hidden and kept only in memory.
hf_token = getpass.getpass("Hugging Face access token: ")

In [ ]:
import requests
r = requests.get(
    "https://huggingface.co/api/models/google/embeddinggemma-300m",
    headers={"Authorization": f"Bearer {hf_token}"}
)
print(r.status_code)  # 200 = OK, 403 = no access

## Container and model configuration

We use the **`server-sagemaker-cuda`** variant of the vLLM Deep Learning Container, which listens on port 8080 and exposes an OpenAI-compatible `/invocations` route (as well as `/ping`). Model configuration is passed entirely through environment variables — no custom inference code required.

Update the `CONTAINER_VERSION` tag below if a newer vLLM DLC has been released; see the [available images reference](https://aws.github.io/deep-learning-containers/reference/available_images/#vllm-ubuntu).

In [ ]:
model_id = "google/embeddinggemma-300m"

# Latest SageMaker vLLM container (update the tag as new versions are released)
CONTAINER_VERSION = "0.25.1-gpu-py312-cu130-ubuntu22.04-sagemaker"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:{CONTAINER_VERSION}"

# EmbeddingGemma is a 300M param model, a single small GPU instance is plenty
instance_type = "ml.g5.xlarge"
# GPU vLLM containers require an inference AMI with compatible NVIDIA drivers
inference_ami_version = "al2-ami-sagemaker-inference-gpu-3-1"

timestamp = time.strftime("%y%m%d-%H%M%S")
model_name = f"embeddinggemma-{timestamp}"
endpoint_config_name = model_name
endpoint_name = model_name
variant_name = "AllTraffic"

print(f"Image: {inference_image}")
print(f"Instance type: {instance_type}")
print(f"Endpoint name: {endpoint_name}")

### vLLM environment variables

- `HF_MODEL_ID` / `HF_TOKEN`: fetch the gated model from the Hugging Face Hub at startup.
- `SM_VLLM_RUNNER=pooling`: EmbeddingGemma's `Gemma3TextModel` architecture is registered in vLLM as a pooling (embedding) model, so the runner must be `pooling` rather than the default text-generation runner.
- `SM_VLLM_MAX_MODEL_LEN=2048`: matches EmbeddingGemma's trained context length.
- `SM_VLLM_GPU_MEMORY_UTILIZATION`: lowered since a 300M model needs very little KV cache / weight memory, leaving headroom on shared instances.

Every `SM_VLLM_<NAME>` variable is translated by the container entrypoint into the matching `--<name>` vLLM server flag.

In [ ]:
env = {
    "HF_MODEL_ID": model_id,
    "HF_TOKEN": hf_token,
    "SM_VLLM_RUNNER": "pooling",
    "SM_VLLM_MAX_MODEL_LEN": "2048",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.85",
    "SM_VLLM_DTYPE": "bfloat16",  # EmbeddingGemma activations do not support float16
}

### Clean up any existing resources with the same names

In [ ]:
for delete_fn, name, label in [
    (sm.delete_endpoint, endpoint_name, "endpoint"),
    (sm.delete_endpoint_config, endpoint_config_name, "endpoint config"),
    (sm.delete_model, model_name, "model"),
]:
    try:
        kwarg_name = {
            sm.delete_endpoint: "EndpointName",
            sm.delete_endpoint_config: "EndpointConfigName",
            sm.delete_model: "ModelName",
        }[delete_fn]
        delete_fn(**{kwarg_name: name})
        print(f"✅ Deleted existing {label}: {name}")
    except sm.exceptions.ClientError as e:
        if "does not exist" in str(e) or "Could not find" in str(e):
            print(f"ℹ️  No existing {label} to delete: {name}")
        else:
            print(f"⚠️  Error deleting {label}: {e}")

ℹ️  No existing endpoint to delete: embeddinggemma-260724-173046
ℹ️  No existing endpoint config to delete: embeddinggemma-260724-173046


ℹ️  No existing model to delete: embeddinggemma-260724-173046


## Create model, endpoint config, and endpoint

In [ ]:
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)
print(f"✅ Created model: {model_name}")

✅ Created model: embeddinggemma-260724-173046


In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance_type,
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": 600,
            # Required for GPU vLLM containers: the default host AMI's NVIDIA
            # drivers are incompatible with this CUDA version.
            "InferenceAmiVersion": inference_ami_version,
        },
    ],
)
print(f"✅ Created endpoint config: {endpoint_config_name}")

✅ Created endpoint config: embeddinggemma-260724-173046


In [ ]:
sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
wait_for_endpoint(endpoint_name)

Waiting for 'embeddinggemma-260724-173046': .........................
Endpoint 'embeddinggemma-260724-173046' status: 'InService'


'InService'

## Run inference

The vLLM SageMaker container exposes an `/invocations` route that follows the [OpenAI Embeddings API](https://platform.openai.com/docs/api-reference/embeddings/create) request/response schema. `invoke_endpoint` sends the JSON body straight through.

EmbeddingGemma expects **task-specific prompt prefixes** to get optimized embeddings (unlike Sentence Transformers, vLLM does not add these automatically — you must prepend them yourself):

| Use case | Prefix |
|---|---|
| Query (retrieval) | `task: search result \| query: {text}` |
| Document (retrieval) | `title: none \| text: {text}` |
| Question answering | `task: question answering \| query: {text}` |
| Classification | `task: classification \| query: {text}` |
| Semantic similarity | `task: sentence similarity \| query: {text}` |

See the [model card](https://huggingface.co/google/embeddinggemma-300m#prompt-instructions) for the full list.

In [ ]:
def embed(texts, dimensions=None):
    payload = {"model": model_id, "input": texts}
    if dimensions is not None:
        payload["dimensions"] = dimensions

    response = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    body = json.loads(response["Body"].read())
    return [item["embedding"] for item in body["data"]]


query_prefix = "task: search result | query: "
document_prefix = "title: none | text: "

query = query_prefix + "Which planet is known as the Red Planet?"
documents = [
    document_prefix + "Venus is often called Earth's twin because of its similar size and proximity.",
    document_prefix + "Mars, known for its reddish appearance, is often referred to as the Red Planet.",
    document_prefix + "Jupiter, the largest planet in our solar system, has a prominent red spot.",
    document_prefix + "Saturn, famous for its rings, is sometimes mistaken for the Red Planet.",
]

start = time.time()
embeddings = embed([query] + documents)
print(f"✅ Got {len(embeddings)} embeddings of dimension {len(embeddings[0])} in {time.time() - start:.2f}s")

✅ Got 5 embeddings of dimension 768 in 0.17s


In [ ]:
import numpy as np

query_embedding = np.array(embeddings[0])
document_embeddings = np.array(embeddings[1:])

# Embeddings returned by the model are already L2-normalized, so the dot
# product is equivalent to cosine similarity.
similarities = query_embedding @ document_embeddings.T
ranking = similarities.argsort()[::-1]

print("Similarities:", np.round(similarities, 4))
print("Ranking (best match first):")
for idx in ranking:
    print(f"  [{similarities[idx]:.4f}] {documents[idx][len(document_prefix):]}")

Similarities: [0.3013 0.636  0.4939 0.4887]
Ranking (best match first):
  [0.6360] Mars, known for its reddish appearance, is often referred to as the Red Planet.
  [0.4939] Jupiter, the largest planet in our solar system, has a prominent red spot.
  [0.4887] Saturn, famous for its rings, is sometimes mistaken for the Red Planet.
  [0.3013] Venus is often called Earth's twin because of its similar size and proximity.


## Truncated embeddings with Matryoshka Representation Learning (MRL)

EmbeddingGemma was trained with MRL, meaning the 768-dim output can theoretically be truncated to smaller dimensions with minimal quality loss.

**Note:** The `dimensions` parameter (server-side truncation via the OpenAI-compatible API) is not yet supported by this vLLM version for EmbeddingGemma. The cell below demonstrates **client-side truncation** instead — slice the full embedding vector and re-normalize. This produces equivalent results to server-side truncation.

In [ ]:
import numpy as np

def truncate_embedding(embedding, dim):
    """Truncate to dim dimensions and re-normalize (MRL client-side)."""
    vec = np.array(embedding[:dim], dtype=np.float32)
    norm = np.linalg.norm(vec)
    return (vec / norm).tolist() if norm > 0 else vec.tolist()

# Get full 768-dim embeddings (no dimensions param — not supported by this vLLM version)
[full_embedding] = embed([query])
print(f"Full embedding dimension: {len(full_embedding)}")

# Client-side truncation via MRL — slice + renormalize
print("\nClient-side MRL truncation:")
for dim in (768, 512, 256, 128):
    truncated = truncate_embedding(full_embedding, dim)
    print(f"  dimensions={dim:>3} -> len(embedding)={len(truncated)}")

Full embedding dimension: 768

Client-side MRL truncation:
  dimensions=768 -> len(embedding)=768
  dimensions=512 -> len(embedding)=512
  dimensions=256 -> len(embedding)=256
  dimensions=128 -> len(embedding)=128


## Cleanup

Delete the endpoint, endpoint config, and model to avoid ongoing charges.

In [ ]:
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_model(ModelName=model_name)
print("✅ Cleaned up endpoint, endpoint config, and model")